# turbopuffer Vector Store

This notebook shows how to use [turbopuffer](https://turbopuffer.com) as a vector store with LlamaIndex.

turbopuffer is a fast, cost-efficient vector database for search and retrieval.

## Setup

Install the required packages:

In [ ]:
! pip install -qU llama-index llama-index-vector-stores-turbopuffer turbopuffer

### Credentials

Create a turbopuffer account at [turbopuffer.com](https://turbopuffer.com) and get an API key.

In [ ]:
import getpass
import os

if not os.getenv("TURBOPUFFER_API_KEY"):
    os.environ["TURBOPUFFER_API_KEY"] = getpass.getpass(
        "Enter your turbopuffer API key: "
    )

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

## Download Sample Data

In [ ]:
! mkdir -p 'data/paul_graham/'
! wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/paul_graham/").load_data()
print("# Documents:", len(documents))

## Create Index with turbopuffer

Initialize the turbopuffer client with a [region](https://turbopuffer.com/docs/regions), create a namespace, and build an index from the documents.

In [ ]:
from turbopuffer import Turbopuffer
from llama_index.vector_stores.turbopuffer import TurbopufferVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

tpuf = Turbopuffer(region="gcp-us-central1")
ns = tpuf.namespace("llamaindex-demo")

vector_store = TurbopufferVectorStore(namespace=ns)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

## Query the Index

In [ ]:
query_engine = index.as_query_engine()

response = query_engine.query("What did the author do growing up?")
print(response)

In [ ]:
response = query_engine.query("What is the author's opinion on startups?")
print(response)

## Metadata Filtering

You can pass `MetadataFilters` to filter results from the turbopuffer vector store.

See the [turbopuffer filter documentation](https://turbopuffer.com/docs/reference/query#filter-parameters) for the full list of supported filter operators.

In [ ]:
from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
    FilterCondition,
)

filters = MetadataFilters(
    filters=[
        MetadataFilter(
            key="theme",
            value=["Fiction", "Horror"],
            operator=FilterOperator.IN,
        ),
        MetadataFilter(key="year", value=1997, operator=FilterOperator.GT),
    ],
    condition=FilterCondition.AND,
)

retriever = index.as_retriever(filters=filters)
retriever.retrieve("What is this about?")

## Hybrid Search (BM25 + Vector)

turbopuffer supports BM25 full-text search natively. You can combine it with vector search using hybrid mode. The `alpha` parameter controls the weighting: 0 = pure BM25, 1 = pure vector, 0.5 = equal blend.

In [ ]:
from llama_index.core.embeddings import resolve_embed_model
from llama_index.core.vector_stores.types import (
    VectorStoreQuery,
    VectorStoreQueryMode,
)

embed_model = resolve_embed_model()

query_text = "What did the author do?"
query = VectorStoreQuery(
    query_embedding=embed_model.get_query_embedding(query_text),
    query_str=query_text,
    similarity_top_k=5,
    mode=VectorStoreQueryMode.HYBRID,
    alpha=0.5,
)
result = vector_store.query(query)
for node in result.nodes:
    print(node.get_content()[:100])

## Cleanup

Delete all vectors in the namespace when you're done.

In [ ]:
vector_store.clear()